In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
import glob, re

main_dir = "../data/main"

# Load all cleaned snapshots sorted chronologically
def _date_key(path):
    # New stamps are YYYYMMDD (from dated snapshots); legacy stamps are MMYYYY.
    m = re.search(r'_(\d{8})_cleaned', path)
    if m:
        return int(m.group(1))
    m = re.search(r'_(\d{2})(\d{4})_cleaned', path)
    if m:
        return int(m.group(2) + m.group(1) + '00')
    return 0

snapshot_files = sorted(
    glob.glob(os.path.join(main_dir, "edc_artist_stats_*_cleaned.csv")),
    key=_date_key
)
print(f"Found {len(snapshot_files)} snapshot(s): {[f.split('/')[-1] for f in snapshot_files]}")

snapshots = [pd.read_csv(f) for f in snapshot_files]
artists_stats = snapshots[-1].copy()  # most recent = primary

# Compute follower/stream growth vs. previous snapshot (% change)
if len(snapshots) >= 2:
    df_prev = snapshots[-2][['artist', 'followers', 'streams']]
    artists_stats = artists_stats.merge(df_prev, on='artist', suffixes=('', '_prev'), how='left')
    artists_stats['followers_growth'] = (
        (artists_stats['followers'] - artists_stats['followers_prev'])
        / artists_stats['followers_prev'].replace(0, np.nan)
    ).fillna(0).round(4)
    artists_stats['streams_growth'] = (
        (artists_stats['streams'] - artists_stats['streams_prev'])
        / artists_stats['streams_prev'].replace(0, np.nan)
    ).fillna(0).round(4)
    artists_stats = artists_stats.drop(columns=['followers_prev', 'streams_prev'])
    print("Growth features computed from 2 snapshots.")
else:
    artists_stats['followers_growth'] = 0.0
    artists_stats['streams_growth'] = 0.0
    print("Only 1 snapshot — growth features default to 0.0.")

# Auto-detect the most recent artist_counts file (e.g., artist_counts_2022_2026.csv)
count_files = sorted(glob.glob(os.path.join(main_dir, "artist_counts_*.csv")))
if not count_files:
    raise FileNotFoundError("No artist_counts_*.csv found in data/main/")
print(f"Loading: {count_files[-1].split('/')[-1]}")
artists_count = pd.read_csv(count_files[-1])
artists_agencies = pd.read_csv(os.path.join(main_dir, "artists_agency.csv"))

Found 1 snapshot(s): ['edc_artist_stats_20260200_cleaned.csv']
Only 1 snapshot — growth features default to 0.0.
Loading: artist_counts_2022_2025.csv


In [3]:
merged_stats_count = pd.merge(artists_stats, artists_count, on="artist", how="left")
merged_all = pd.merge(merged_stats_count, artists_agencies, on="artist", how="left")
merged_all.head()

,artist,followers,streams,playlists,playlist reach,charts,shazams,videos,views,dj supports,followers_growth,streams_growth,total_appearances,years_played,agency
0,1080p,6,5986,1,641,0,17,45,9800,0,0.0,0.0,1.0,[2024],insomniac
1,a shade of black,0,15100,0,11000,7,0,0,0,0,0.0,0.0,1.0,[2024],NaN
2,aaron k,1,2163,0,0,1,4,2,0,0,0.0,0.0,2.0,"[2024, 2025]",insomniac
3,abana,7710,468000,175,4220000,60,15300,110,1890000,93,0.0,0.0,3.0,"[2022, 2023, 2024]",insomniac
4,d. zeledon,5776,69600,25,218000,18,764,31,9672,58,0.0,0.0,1.0,[2024],insomniac


In [4]:
# Convert total_appearances to int, replace 0 with 1
merged_all['total_appearances'] = merged_all['total_appearances'].fillna(0).astype(int)
merged_all['total_appearances'] = merged_all['total_appearances'].replace(0, 1)

# Clean years_played to show only years (e.g., "2024, 2025" instead of "[np.int64(2024)]")
def clean_years(val):
    if pd.isna(val) or val == 0:
        return ""
    val_str = str(val)
    # Extract all 4-digit years from the string
    import re
    years = re.findall(r'\b(20\d{2})\b', val_str)
    return ', '.join(years) if years else val_str

merged_all['years_played'] = merged_all['years_played'].apply(clean_years).astype(str)

merged_all.head()

,artist,followers,streams,playlists,playlist reach,charts,shazams,videos,views,dj supports,followers_growth,streams_growth,total_appearances,years_played,agency
0,1080p,6,5986,1,641,0,17,45,9800,0,0.0,0.0,1,2024,insomniac
1,a shade of black,0,15100,0,11000,7,0,0,0,0,0.0,0.0,1,2024,NaN
2,aaron k,1,2163,0,0,1,4,2,0,0,0.0,0.0,2,"2024, 2025",insomniac
3,abana,7710,468000,175,4220000,60,15300,110,1890000,93,0.0,0.0,3,"2022, 2023, 2024",insomniac
4,d. zeledon,5776,69600,25,218000,18,764,31,9672,58,0.0,0.0,1,2024,insomniac


In [5]:
merged_all = merged_all.drop_duplicates()
merged_all.head()

,artist,followers,streams,playlists,playlist reach,charts,shazams,videos,views,dj supports,followers_growth,streams_growth,total_appearances,years_played,agency
0,1080p,6,5986,1,641,0,17,45,9800,0,0.0,0.0,1,2024,insomniac
1,a shade of black,0,15100,0,11000,7,0,0,0,0,0.0,0.0,1,2024,NaN
2,aaron k,1,2163,0,0,1,4,2,0,0,0.0,0.0,2,"2024, 2025",insomniac
3,abana,7710,468000,175,4220000,60,15300,110,1890000,93,0.0,0.0,3,"2022, 2023, 2024",insomniac
4,d. zeledon,5776,69600,25,218000,18,764,31,9672,58,0.0,0.0,1,2024,insomniac


In [6]:
# Convert columns to numeric values
numeric_cols = ['followers', 'streams', 'playlists', 'playlist reach', 
                'charts', 'shazams', 'videos', 'views', 'dj supports', 'total_appearances']

for col in numeric_cols:
    merged_all[col] = pd.to_numeric(merged_all[col], errors='coerce').fillna(0).astype(int)

# Verify the conversions
print(merged_all[numeric_cols].dtypes)

followers            int64
streams              int64
playlists            int64
playlist reach       int64
charts               int64
shazams              int64
videos               int64
views                int64
dj supports          int64
total_appearances    int64
dtype: object


In [7]:
merged_all.drop_duplicates()
# merged_all.count

,artist,followers,streams,playlists,playlist reach,charts,shazams,videos,views,dj supports,followers_growth,streams_growth,total_appearances,years_played,agency
0,1080p,6,5986,1,641,0,17,45,9800,0,0.0,0.0,1,2024,insomniac
1,a shade of black,0,15100,0,11000,7,0,0,0,0,0.0,0.0,1,2024,NaN
2,aaron k,1,2163,0,0,1,4,2,0,0,0.0,0.0,2,"2024, 2025",insomniac
3,abana,7710,468000,175,4220000,60,15300,110,1890000,93,0.0,0.0,3,"2022, 2023, 2024",insomniac
4,d. zeledon,5776,69600,25,218000,18,764,31,9672,58,0.0,0.0,1,2024,insomniac
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1383,decoder,32299,113000,0,147000,27,0,0,22800,84,0.0,0.0,1,,insomniac
1384,4b,706000,201000000,0,70100000,424,604000,1210000,363000000,6272,0.0,0.0,1,,insomniac
1385,4b,706000,201000000,0,70100000,424,604000,1210000,363000000,6272,0.0,0.0,1,,wasserman
1386,malaa,1220000,3100000000,10800,148000000,876,3170000,63500,682000000,10500,0.0,0.0,1,2023,insomniac


In [8]:
merged_all.to_csv("../data/main/COMPLETE_edc_artist_and_stats.csv", index=False)

In [9]:
# Check dataframe shape and content
print(f"Rows: {len(merged_all)}, Columns: {len(merged_all.columns)}")
merged_all.head()

Rows: 1388, Columns: 15


,artist,followers,streams,playlists,playlist reach,charts,shazams,videos,views,dj supports,followers_growth,streams_growth,total_appearances,years_played,agency
0,1080p,6,5986,1,641,0,17,45,9800,0,0.0,0.0,1,2024,insomniac
1,a shade of black,0,15100,0,11000,7,0,0,0,0,0.0,0.0,1,2024,NaN
2,aaron k,1,2163,0,0,1,4,2,0,0,0.0,0.0,2,"2024, 2025",insomniac
3,abana,7710,468000,175,4220000,60,15300,110,1890000,93,0.0,0.0,3,"2022, 2023, 2024",insomniac
4,d. zeledon,5776,69600,25,218000,18,764,31,9672,58,0.0,0.0,1,2024,insomniac


In [10]:
# Check the data type of years_played
print(merged_all['years_played'].dtype)
print(merged_all['years_played'].head(10))

object
0                2024
1                2024
2          2024, 2025
3    2022, 2023, 2024
4                2024
5    2022, 2023, 2024
6    2022, 2023, 2024
7    2022, 2023, 2024
8    2022, 2023, 2024
9          2022, 2024
Name: years_played, dtype: object


In [11]:
# Remove duplicates where an artist has multiple entries and one is 'insomniac'
# Logic: If duplicates exist on 'artist', prefer the non-insomniac agency.

# 1. Identify duplicates based on 'artist'
duplicates_mask = merged_all.duplicated(subset=['artist'], keep=False)

# 2. Split into duplicates and non-duplicates
duplicates_df = merged_all[duplicates_mask]
non_duplicates_df = merged_all[~duplicates_mask]

# 3. Filter duplicates: remove 'insomniac' if another agency exists
# We can sort by agency so that 'insomniac' is prioritized for removal if we use drop_duplicates
# Or we can explicitly filter.

# Let's verify if 'agency' column exists and clean it
if 'agency' in merged_all.columns:
    # Normalized check
    duplicates_df = duplicates_df.copy() # Avoid SettingWithCopyWarning
    
    # Custom sort: make 'insomniac' appearing at the end or beginning to control drop_duplicates
    # If we want to KEEP non-insomniac, we want non-insomniac to be first.
    # So we can assign a priority: non-insomniac = 0, insomniac = 1. sort, then keep='first'
    
    def agency_priority(agency):
        if str(agency).lower().strip() == 'insomniac':
            return 1
        return 0
    
    duplicates_df['priority'] = duplicates_df['agency'].apply(agency_priority)
    duplicates_df = duplicates_df.sort_values('priority')
    
    # Keep the first one (which will be priority 0 / non-insomniac if available)
    deduped_duplicates = duplicates_df.drop_duplicates(subset=['artist'], keep='first')
    
    # Drop helper column
    deduped_duplicates = deduped_duplicates.drop(columns=['priority'])
    
    # Combine back
    merged_all = pd.concat([non_duplicates_df, deduped_duplicates], ignore_index=True)

print(f"Rows after removing insomniac duplicates: {len(merged_all)}")
merged_all[merged_all['artist'] == 'kaskade']

Rows after removing insomniac duplicates: 973


,artist,followers,streams,playlists,playlist reach,charts,shazams,videos,views,dj supports,followers_growth,streams_growth,total_appearances,years_played,agency
955,kaskade,8720000,3870000000,33500,481000000,2149,21100000,728000,1450000000,18500,0.0,0.0,4,"2022, 2023, 2024, 2025",uta


In [12]:

output_path = "../data/main/COMPLETE_edc_artist_and_stats.csv"
merged_all.to_csv(output_path, index=False)
print(f"Saved cleaned data to {output_path}")

Saved cleaned data to ../data/main/COMPLETE_edc_artist_and_stats.csv


some artist are duplicate because they have 2 agencies, insomniac and another one. Delete the dup that has agency = "insomniac"